## A notebook for a comprehensive look at the synoptic forecast

Import all used libraries

In [ ]:
import glob, base64, json, pathlib, re, tempfile, uuid

import numpy as np
import pandas as pd
import xarray as xr
import metpy 
from metpy.units import units

import earthkit.data as ekd
import earthkit.plots as ekp
import matplotlib.pyplot as plt
import matplotlib as mpl
import cmcrameri.cm as cm

import ipywidgets as widgets
from IPython.display import HTML, Image, display

Register the crameri colour maps to matplotlib

In [ ]:
mpl.colormaps.register(cm.tokyo)
mpl.colormaps.register(cm.batlowW)

The HTML template to display the maps

In [ ]:
TEMPLATE = '''
<div id="ek-__UID__" style="font-family:sans-serif">
  <div style="display:flex;align-items:center;gap:10px;margin-bottom:6px">
    <button id="play-__UID__" style="width:36px;height:28px;cursor:pointer">&#9654;</button>
    <input id="slider-__UID__" type="range" min="0" max="__MAX__" value="0" style="width:500px">
    <span id="label-__UID__" style="min-width:200px"></span>
  </div>
  <img id="img-__UID__" style="max-width:100%;height:auto">
</div>
<script>
(function () {
  const frames = __FRAMES__;
  const labels = __LABELS__;
  const interval = __INTERVAL__;
  const img = document.getElementById("img-__UID__");
  const slider = document.getElementById("slider-__UID__");
  const label = document.getElementById("label-__UID__");
  const play = document.getElementById("play-__UID__");
  frames.forEach(s => { const im = new Image(); im.src = s; });   // warm the cache

  function show(i) {
    img.src = frames[i];
    label.textContent = labels[i];
    slider.value = i;
  }
  slider.addEventListener("input", () => show(+slider.value));    // live while dragging

  let timer = null;
  play.addEventListener("click", () => {
    if (timer) { clearInterval(timer); timer = null; play.innerHTML = "&#9654;"; return; }
    play.innerHTML = "&#10074;&#10074;";
    timer = setInterval(() => show((+slider.value + 1) % frames.length), interval);
  });
  show(0);
})();
</script>
'''

Decide the area to look at and load the IFS forecast data

In [ ]:
DOMAIN = "North Atlantic"

# GRIB via earthkit's xarray engine gives the dims: step, level (hPa), latitude, longitude.
# valid_time(step) is added on request; the run time is kept as a scalar coordinate.
LEV = "level"

# Box around DOMAIN to keep in memory (lon_min, lon_max, lat_min, lat_max); None = global.
# Same as CROP_BOX in build_site.py.
CROP_BOX = (-110, 95, 10, 90)

def open_grib(path, pressure_levels=False):
    if pressure_levels:
        # squeeze=False keeps "level" even in single-level files like pl_850.grib
        ds = ekd.from_source("file", str(path)).to_xarray(add_valid_time_coord=True, squeeze=False)
        ds = ds.isel(forecast_reference_time=-1, level_type=0).drop_vars("level_type")
    else:
        ds = ekd.from_source("file", str(path)).to_xarray(add_valid_time_coord=True)
        if "forecast_reference_time" in ds.dims:     # several runs in the file: keep the latest
            ds = ds.isel(forecast_reference_time=-1)

    # Crop first, then read into memory as float32 (global float64 needs ~5 GB)
    ds = crop(fix_units(ds)).load()
    return ds.astype("float32", keep_attrs=True)

def fix_units(ds):
    """earthkit-data >= 1.0 writes units like "m.s**-1", which pint/MetPy can't parse."""
    for v in ds.data_vars:
        u = ds[v].attrs.get("units")
        if isinstance(u, str):
            ds[v].attrs["units"] = re.sub(r"(?<=[A-Za-z0-9])\.(?=[A-Za-z])", " ", u)
    return ds

def crop(ds):
    if CROP_BOX is None:
        return ds
    lon_min, lon_max, lat_min, lat_max = CROP_BOX
    if float(ds.longitude.max()) > 180:          # 0..360 grid -> -180..180
        ds = ds.assign_coords(longitude=((ds.longitude + 180) % 360) - 180).sortby("longitude")
    lat = slice(lat_max, lat_min) if ds.latitude[0] > ds.latitude[-1] else slice(lat_min, lat_max)
    return ds.sel(longitude=slice(lon_min, lon_max), latitude=lat)

# Pressure levels come in one file per level group (pl_500-1000.grib, pl_850.grib, ...),
# because earthkit won't put variables with different levels into one dataset.
# Merging puts every variable on the union of levels (NaN where not downloaded).
pl_files = sorted(glob.glob("../Data/pl_*.grib"))
data = xr.merge([open_grib(f, pressure_levels=True) for f in pl_files]
                + [open_grib("../Data/sfc.grib")],
                compat="override", join="outer")

N_STEPS = data.sizes["step"]
data

Define all the maps plotted

In [ ]:
def render_SurfaceTemperaturePressure(frame, stamp, fc_step): 

    # do any precalculations necessary 

    temperature = frame["2t"]
    msl = frame["msl"]

    # set up the styles
    t2m_style = ekp.styles.Style(
            units="celsius",
            colors="RdBu_r",
            levels=np.arange(-40,41,2),
            )
    
    msl_style = ekp.styles.Contour(
            units="hPa",
            levels=np.arange(920,1081,4),
            colors="black",
            linewidths=[1.4, 0.8, 0.8, 0.8, 0.8],
            linestyles=["solid",],
            labels=True,
            legend_style=None
            )

    # plot all the variables of interest
    chart = ekp.Map(domain=DOMAIN)
    chart.title(f"{{variable_name}}\n{stamp} (+{fc_step} h)")
    
    

    # Surface temperature
    chart.contourf(temperature, style=t2m_style)

    # Mean sea level pressure
    chart.contour(msl, style=msl_style)

    chart.legend()

    return chart 


def render_ReTop(frame, stamp, fc_step): 

    # do any precalculations necessary 

    gphDiff = (frame["gh"].sel({LEV: 500}) - frame["gh"].sel({LEV: 1000})) 
    gph500 = frame["gh"].sel({LEV: 500})
    msl = frame["msl"]

    # set up the styles
    gphDiff_style = ekp.styles.Style(
            colors="batlowW",
            extend="both", 
            levels=np.arange(4400, 6010, 40),
            )
    
    gph500_style = ekp.styles.Contour(
            levels=np.arange(5000,6000,80),
            colors="black",
            linewidths=[1.4],
            linestyles=["solid",],
            labels=True,
            legend_style=None
            )
    
    msl_style = ekp.styles.Contour(
            units="hPa",
            levels=np.arange(920,1081,5),
            colors="white",
            linewidths=[1.2,],
            linestyles=["solid",],
            labels=True,
            legend_style=None
            )

    # plot all the variables of interest
    chart = ekp.Map(domain=DOMAIN)
    chart.title(f"{{variable_name}}\n{stamp} (+{fc_step} h)")
    

    # Surface temperature
    chart.contourf(gphDiff, style=gphDiff_style)

    # Geopotential height at 500 hPa
    chart.contour(gph500, style=gph500_style)

    # Mean sea level pressure
    chart.contour(msl, style=msl_style)

    chart.legend()

    return chart 


def render_Frontogenesis(frame, stamp, fc_step): 

    msl = frame["msl"].load()

    # do any precalculations necessary 
    frame = frame.sel({LEV: 850}).load()
    pt = metpy.calc.potential_temperature(850 * units.hPa, frame["t"])
    fgp = metpy.calc.frontogenesis(pt, frame["u"], frame["v"]) * 3.6E9
    

    # set up the styles
    fgp_style = ekp.styles.Style(
            units="K/(1000kmh)",
            colors="RdBu_r",
            extend="both",
            levels=np.arange(-15, 15.1, 0.5),
            )
    
    msl_style = ekp.styles.Contour(
            units="hPa",
            levels=np.arange(920,1081,5),
            colors="black",
            linewidths=[1.,],
            linestyles=["solid",],
            labels=True,
            legend_style=None
            )

    # plot all the variables of interest
    chart = ekp.Map(domain=DOMAIN)
    chart.title(f"{{variable_name}}\n{stamp} (+{fc_step} h)")
    

    # Frontogenesis parameter
    chart.contourf(fgp, style=fgp_style)

    # Mean sea level pressure
    chart.contour(msl, style=msl_style)

    chart.legend()

    return chart 


def render_EqPotTemperature(frame, stamp, fc_step): 

    msl = frame["msl"].load()

    # do any precalculations necessary 
    frame = frame.sel({LEV: 850}).load()
    td = metpy.calc.dewpoint_from_specific_humidity(850 * units.hPa, frame["t"],  frame["q"])
    ept = metpy.calc.equivalent_potential_temperature(850 * units.hPa, frame["t"], td).metpy.convert_units("degC")

    # set up the styles
    ept_style = ekp.styles.Style(
            units="degC",
            colors="tokyo",
            extend="both",
            levels=np.arange(-10, 81, 2),
            )
    
    msl_style = ekp.styles.Contour(
            units="hPa",
            levels=np.arange(920,1081,5),
            colors="black",
            linewidths=[1.,],
            linestyles=["solid",],
            labels=True,
            legend_style=None
            )

    # plot all the variables of interest
    chart = ekp.Map(domain=DOMAIN)
    chart.title(f"{{variable_name}}\n{stamp} (+{fc_step} h)")
    

    # Frontogenesis parameter
    chart.contourf(ept, style=ept_style)

    # Mean sea level pressure
    chart.contour(msl, style=msl_style)

    chart.legend()

    return chart 


def render_Tcwv(frame, stamp, fc_step): 

    msl = frame["msl"].load()
    tcwv = frame["tcwv"].load()

    # set up the styles
    tcwv_style = ekp.styles.Style(
            units_label="kg/m2",
            colors="GnBu",
            extend="both",
            levels=np.arange(0, 46, 1),
            )
    
    msl_style = ekp.styles.Contour(
            units="hPa",
            levels=np.arange(920,1081,5),
            colors="black",
            linewidths=[1.,],
            linestyles=["solid",],
            labels=True,
            legend_style=None
            )

    # plot all the variables of interest
    chart = ekp.Map(domain=DOMAIN)
    chart.title(f"{{variable_name}}\n{stamp} (+{fc_step} h)")
    

    # Frontogenesis parameter
    chart.contourf(tcwv, style=tcwv_style)

    # Mean sea level pressure
    chart.contour(msl, style=msl_style)

    chart.legend()

    return chart 


def render_JetDiagnose(frame, stamp, fc_step): 

    msl = frame["msl"]

    # do any precalculations necessary 

    frame = frame.sel({LEV: 200})
    speed = metpy.calc.wind_speed(frame["u"], frame["v"])
    

    # set up the styles
    speed_style = ekp.styles.Style(
            units="m/s",
            colors="YlGn",
            extend="max", 
            levels=np.arange(30,91,5),
            )
    
    msl_style = ekp.styles.Contour(
            units="hPa",
            levels=np.arange(920,1081,4),
            colors="black",
            linewidths=[1.4, 0.8, 0.8, 0.8, 0.8],
            linestyles=["solid",],
            labels=True,
            legend_style=None
            )

    # plot all the variables of interest
    chart = ekp.Map(domain=DOMAIN)
    chart.title(f"{{variable_name}}\n{stamp} (+{fc_step} h)")
    
    

    # Surface temperature
    chart.contourf(speed, style=speed_style)

    # Mean sea level pressure
    chart.contour(msl, style=msl_style)

    chart.legend()

    return chart



def render_frame(figure, i, da, path,):

    # Get the forecast frame, its valid time and lead time
    frame = da.isel(step=i)
    stamp = pd.Timestamp(frame["valid_time"].values).strftime("%d %b %Y, %H:%M UTC")
    fc_step = str(int(frame["step"].values / np.timedelta64(1, "h")))

    match figure:
        
        case "SurfaceTemperaturePressure":
            chart = render_SurfaceTemperaturePressure(frame, stamp, fc_step)

        case "ReTop":
            chart = render_ReTop(frame, stamp, fc_step)

        case "Frontogenesis":
            chart = render_Frontogenesis(frame, stamp, fc_step)
        
        case "EqPotTemperature":
            chart = render_EqPotTemperature(frame, stamp, fc_step)
        
        case "Tcwv":
            chart = render_Tcwv(frame, stamp, fc_step)

        case "JetDiagnose":
            chart = render_JetDiagnose(frame, stamp, fc_step)


    # all the miscellanious
    chart.coastlines()
    chart.borders()
    chart.gridlines()


    # save it to the path and close the plots
    chart.save(str(path))
    plt.close("all")

    return stamp


def RenderProcess(figure): 

    n = N_STEPS
    tmpdir = pathlib.Path(tempfile.mkdtemp())
    pngs, labels = [], []

    bar = widgets.IntProgress(value=0, min=0, max=n, description=f"Rendering {figure}", bar_style="info")
    display(bar)
    for i in range(n):
        p = tmpdir / f"frame_{i:03d}.png"
        labels.append(render_frame(figure, i, data, p))
        pngs.append(p.read_bytes())
        bar.value = i + 1
    bar.bar_style = "success"

    print(f"Rendered {figure} {n} frames, {sum(map(len, pngs)) / 1e6:.1f} MB of PNG data")

    return pngs, labels

def build_viewer(pngs, labels, interval_ms=400):
    srcs = ["data:image/png;base64," + base64.b64encode(b).decode() for b in pngs]
    return (TEMPLATE
            .replace("__UID__", uuid.uuid4().hex[:8])
            .replace("__MAX__", str(len(pngs) - 1))
            .replace("__FRAMES__", json.dumps(srcs))
            .replace("__LABELS__", json.dumps(labels))
            .replace("__INTERVAL__", str(interval_ms)))




Pre-render all the charts so the animations are smooth

In [ ]:
pngs_SurfaceTemperaturePressure, labels_SurfaceTemperaturePressure = RenderProcess("SurfaceTemperaturePressure")
pngs_ReTop, labels_ReTop = RenderProcess("ReTop")
pngs_Frontogenesis, labels_Frontogenesis = RenderProcess("Frontogenesis")
pngs_EqPotTemperature, labels_EqPotTemperature = RenderProcess("EqPotTemperature")
pngs_Tcwv, labels_Tcwv = RenderProcess("Tcwv")
pngs_JetDiagnose, labels_JetDiagnose = RenderProcess("JetDiagnose")

### North Atlantic 2m temperature and MSLP

In [ ]:
viewer_html = build_viewer(pngs_SurfaceTemperaturePressure, labels_SurfaceTemperaturePressure)
display(HTML(viewer_html))

### Large-scale synoptic situation
Shading: Geopotential height difference between 1000 hPa and 500 hP --> Bulk temperature

Black contour: Geopotential height

White contour: MSLP

In [ ]:
viewer_html = build_viewer(pngs_ReTop, labels_ReTop)
display(HTML(viewer_html))

### Pseudo-equivalent potential temperature: Look at airmasses
Shading: Pseudo-equivalent potential temperature

Black contour: MSLP

In [ ]:
viewer_html = build_viewer(pngs_EqPotTemperature, labels_EqPotTemperature)
display(HTML(viewer_html))

### Frontogenesis parameter: Where fronts are forming
Shading: Frontogenesis parameter

Black contour: MSLP

In [ ]:
viewer_html = build_viewer(pngs_Frontogenesis, labels_Frontogenesis)
display(HTML(viewer_html))

### Upper-level jet
Shading: Wind speed at 200 hPa

Black contour: MSLP

In [ ]:
viewer_html = build_viewer(pngs_JetDiagnose, labels_JetDiagnose)
display(HTML(viewer_html))

### Total column water vapour: How wet are we talking
Shading: Total column water

Black contour: MSLP

In [ ]:
viewer_html = build_viewer(pngs_Tcwv, labels_Tcwv)
display(HTML(viewer_html))

### Analysis by DWD

In [ ]:
display(Image("../Data/DWD_analysis/Analysis.png"))

### Analysis forecast by DWD

In [ ]:
files = sorted(glob.glob("../Data/DWD_analysis/Analysis_p*.png"))
dwd_pngs = [open(f, "rb").read() for f in files]

viewer_html = build_viewer(dwd_pngs, [str(k) for k in range(len(dwd_pngs))])
display(HTML(viewer_html))

### Ensemble forecast for Helsinki

In [ ]:
display(Image("../Data/meteograms/helsinki_meteogram.png"))